# Assignment 2 — Image dehazing: 4 models × 3 datasets

**Models:** FFA-Net · MSBDN-DFF · AECR-Net · DEA-Net (official code + official pretrained weights)  
**Datasets:** RESIDE-SOTS indoor (500 pairs) · O-HAZE (45 pairs) · NH-HAZE (55 pairs)  
**Metrics:** PSNR, SSIM, MAE, inference time, parameters, all computed with the same code for every model.

Run the cells from top to bottom. Every step can be re-run safely; finished downloads and predictions are skipped.

## 0. Install packages (first time only)

In [ ]:
import importlib.util, subprocess, sys, shutil
if importlib.util.find_spec("torch") is None:
    if shutil.which("nvidia-smi"):   # NVIDIA GPU -> CUDA build
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision",
                               "--index-url", "https://download.pytorch.org/whl/cu126"])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("packages ready")

## 1. Setup

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path(r"C:\Users\lab\Desktop\dehaze")
if not (ROOT / "dehaze_lib").exists():
    ROOT = Path.cwd()          # notebook opened from the project folder elsewhere
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import torch, pandas as pd
from IPython.display import display, Markdown
from dehaze_lib import config as C
from dehaze_lib.download import download_datasets, download_weights, weights_status
from dehaze_lib.data import organize_all, get_pairs
from dehaze_lib.models import run_inference
from dehaze_lib.evaluate import evaluate_all, plot_results, plot_gain, show_examples

print("Project:", C.ROOT)
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU, slow)")
print("Models  :", C.MODELS)
print("Datasets:", C.DATASETS)

## 2. Download the datasets (≈1.3 GB) and the official weights

In [ ]:
download_datasets()

In [ ]:
download_weights()
pd.Series(weights_status(), name="downloaded")

## 3. Organize into `datasets/<name>/{hazy,gt}`
RESIDE-SOTS indoor has 50 clear scenes × 10 haze levels = 500 hazy images, so each hazy image is matched to its scene's GT.
O-HAZE is downscaled to a maximum side of 1600 px (hazy and GT by the same factor; see `C.MAX_SIDE`).

In [ ]:
organize_all();

## 4. Quick test (3 images per dataset)
Checks that every model loads and runs before starting the long run.

In [ ]:
run_inference(limit=3);

## 5. Full inference
Writes `predictions/<dataset>/<model>/<image>.png`. Takes roughly 15–40 min on a GPU and several hours on CPU.
If you get out-of-memory errors, add `tile=512`.

In [ ]:
log = run_inference()          # e.g. run_inference(models=["DEA-Net"], datasets=["O_HAZE"], tile=512)
pd.read_csv(C.RESULTS / "inference_log.csv")

## 6. Evaluation: PSNR / SSIM / MAE

In [ ]:
summary = evaluate_all()
cols = ["Dataset", "Model", "Checkpoint", "N", "PSNR", "SSIM", "MAE", "Sec_per_image", "Params_M"]  # NH_HAZE_test51-55 = official test split
display(summary[[c for c in cols if c in summary]].round(4))

In [ ]:
for metric in ["PSNR", "SSIM"]:
    display(Markdown(f"**{metric}** (rows = model, columns = dataset)"))
    display(pd.read_csv(C.RESULTS / f"table_{metric}.csv", index_col=0).round(4))

## 7. Figures

In [ ]:
plot_results(summary)
plot_gain(summary);

In [ ]:
for ds in C.DATASETS:
    show_examples(ds, idx=0)      # change idx to see other images

## 8. Notes for the report
* **Checkpoints:** FFA-Net, AECR-Net and DEA-Net use their RESIDE-ITS (indoor) weights on SOTS-indoor. MSBDN-DFF uses its single RESIDE model.
  For O-HAZE and NH-HAZE no author released O-HAZE weights, so FFA-Net and DEA-Net use their OTS (outdoor) weights,
  MSBDN-DFF its RESIDE weights, and AECR-Net its NH-HAZE weights (AECR-Net on NH-HAZE is therefore in-domain, the rest is cross-dataset).
* **Fair comparison:** same test images, same pre-processing, same metric code (scikit-image, RGB, Gaussian SSIM).
* **Differences from the papers** are expected: different SSIM implementations (MATLAB vs Python), O-HAZE resizing, and cross-dataset checkpoints.
* **AECR-Net:** the CUDA-only DCNv2 layer was replaced by `torchvision.ops.deform_conv2d` (the same modulated deformable convolution) so it runs on Windows.
* All files: `results/summary_all.csv`, `results/summary.md`, `results/table_*.csv`, `results/per_image/`, `results/figures/`.